In [0]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 13.0827,
    "longitude": 80.2707,
    "start_date": "2026-07-01",
    "end_date": "2026-07-02",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
        "weather_code",
        "wind_speed_10m"
    ],
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=30)

response.raise_for_status()

data = response.json()

print("API status:", response.status_code)
print("Available fields:", list(data["hourly"].keys()))
print("Number of timestamps:", len(data["hourly"]["time"]))

API status: 200
Available fields: ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'weather_code', 'wind_speed_10m']
Number of timestamps: 48


In [0]:
from pyspark.sql import Row

# Convert API data into rows
rows = [
    Row(
        time=data["hourly"]["time"][i],
        temperature_2m=data["hourly"]["temperature_2m"][i],
        relative_humidity_2m=data["hourly"]["relative_humidity_2m"][i],
        precipitation=data["hourly"]["precipitation"][i],
        rain=data["hourly"]["rain"][i],
        weather_code=data["hourly"]["weather_code"][i],
        wind_speed_10m=data["hourly"]["wind_speed_10m"][i]
    )
    for i in range(len(data["hourly"]["time"]))
]

# Create Spark DataFrame
df = spark.createDataFrame(rows)

print("Total rows:", df.count())
print("Total columns:", len(df.columns))

display(df)

Total rows: 48
Total columns: 7


time,temperature_2m,relative_humidity_2m,precipitation,rain,weather_code,wind_speed_10m
2026-07-01T00:00,31.1,55,0.0,0.0,3,17.8
2026-07-01T01:00,30.5,56,0.0,0.0,3,17.4
2026-07-01T02:00,29.9,57,0.0,0.0,3,18.1
2026-07-01T03:00,29.5,58,0.0,0.0,3,16.8
2026-07-01T04:00,29.1,59,0.0,0.0,3,16.4
2026-07-01T05:00,28.7,64,0.0,0.0,3,14.5
2026-07-01T06:00,28.7,64,0.0,0.0,3,14.6
2026-07-01T07:00,29.4,63,0.0,0.0,3,13.6
2026-07-01T08:00,30.5,60,0.0,0.0,3,15.3
2026-07-01T09:00,31.6,56,0.0,0.0,3,18.1


In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_pipeline.bronze.weather_test")

In [0]:
bronze_df = spark.table("my_pipeline.bronze.weather_test")

print("Bronze rows:", bronze_df.count())
bronze_df.printSchema()

Bronze rows: 48
root
 |-- time: string (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)



In [0]:
from pyspark.sql.functions import col, to_timestamp

# Read Bronze
bronze_df = spark.table("my_pipeline.bronze.weather_test")

# Convert time to timestamp
silver_df = bronze_df.withColumn(
    "time", to_timestamp(col("time"))
)

# Remove duplicate timestamps
silver_df = silver_df.dropDuplicates(["time"])

# Remove rows with missing timestamps
silver_df = silver_df.filter(col("time").isNotNull())

# Display the cleaned data
print("Silver rows:", silver_df.count())
silver_df.printSchema()
display(silver_df)

Silver rows: 48
root
 |-- time: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)



time,temperature_2m,relative_humidity_2m,precipitation,rain,weather_code,wind_speed_10m
2026-07-01T00:00:00.000Z,31.1,55,0.0,0.0,3,17.8
2026-07-01T01:00:00.000Z,30.5,56,0.0,0.0,3,17.4
2026-07-01T02:00:00.000Z,29.9,57,0.0,0.0,3,18.1
2026-07-01T03:00:00.000Z,29.5,58,0.0,0.0,3,16.8
2026-07-01T04:00:00.000Z,29.1,59,0.0,0.0,3,16.4
2026-07-01T05:00:00.000Z,28.7,64,0.0,0.0,3,14.5
2026-07-01T06:00:00.000Z,28.7,64,0.0,0.0,3,14.6
2026-07-01T07:00:00.000Z,29.4,63,0.0,0.0,3,13.6
2026-07-01T08:00:00.000Z,30.5,60,0.0,0.0,3,15.3
2026-07-01T09:00:00.000Z,31.6,56,0.0,0.0,3,18.1


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_pipeline.silver.weather_test")

In [0]:
silver_check = spark.table("my_pipeline.silver.weather_test")

print("Silver rows:", silver_check.count())
silver_check.printSchema()

Silver rows: 48
root
 |-- time: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)



In [0]:
tables = [
    "my_pipeline.bronze.weather",
    "my_pipeline.silver.weather",
    "my_pipeline.gold.weather"
]

for table in tables:
    df = spark.table(table)
    print(f"{table}: {df.count()} rows")

my_pipeline.bronze.weather: 4512 rows
my_pipeline.silver.weather: 4512 rows
my_pipeline.gold.weather: 181 rows


In [0]:
bronze_df = spark.table("my_pipeline.bronze.weather")

print("Total rows:", bronze_df.count())
print("Unique timestamps:",
      bronze_df.select("time").distinct().count())

Total rows: 4512
Unique timestamps: 4512


In [0]:
silver_df = spark.table("my_pipeline.silver.weather")

print("Total rows:", silver_df.count())
print("Unique timestamps:",
      silver_df.select("time").distinct().count())

silver_df.printSchema()

Total rows: 4512
Unique timestamps: 4512
root
 |-- time: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: long (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- location: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
gold_df = spark.table("my_pipeline.gold.weather")

print("Total rows:", gold_df.count())
print("Unique dates:",
      gold_df.select("date").distinct().count())

gold_df.printSchema()

Total rows: 181
Unique dates: 181
root
 |-- date: date (nullable = true)
 |-- avg_temperature: double (nullable = true)
 |-- min_temperature: double (nullable = true)
 |-- max_temperature: double (nullable = true)
 |-- avg_apparente_temperature: double (nullable = true)
 |-- avg_relative_humidity: double (nullable = true)
 |-- total_precipation: double (nullable = true)
 |-- total_rain: double (nullable = true)
 |-- max_wind_speed_10m: double (nullable = true)
 |-- max_wind_guests_10m: double (nullable = true)
 |-- avg_cloud_cover: double (nullable = true)



In [0]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 13.0827,
    "longitude": 80.2707,
    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "wind_speed_10m,"
        "weather_code,"
        "rain,"
        "apparent_temperature,"
        "precipitation,"
        "surface_pressure,"
        "cloud_cover,"
        "wind_direction_10m,"
        "wind_gusts_10m"
    ),
    "forecast_days": 16,
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()
print("API status:", response.status_code)
print("Available hourly records:", len(data["hourly"]["time"]))

API status: 200
Available hourly records: 384


---------------------------------------------------------------------------
PySparkTypeError                          Traceback (most recent call last)
File <command-4839411214953157>, line 11
      4 existing_times = {
      5     str(row["time"])[:16]
      6     for row in existing_df.select("time").collect()
      7 }
      9 new_times = data["hourly"]["time"]
---> 11 new_count = sum(
     12     1 for t in new_times
     13     if t[:16] not in existing_times
     14 )
     16 print("Fetched records:", len(new_times))
     17 print("New timestamps:", new_count)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/utils.py:310, in try_remote_functions.<locals>.wrapped(*args, **kwargs)
    307 if is_remote() and "PYSPARK_NO_NAMESPACE_SHARE" not in os.environ:
    308     from pyspark.sql.connect import functions
--> 310     return getattr(functions, f.__name__)(*args, **kwargs)
    311 else:
    312     return f(*args, **kwargs)

File /databricks/python/lib/python3.12/s

In [0]:
print(type(data))
print(type(data["hourly"]))
print(type(data["hourly"]["time"]))

<class 'dict'>
<class 'dict'>
<class 'list'>


In [0]:
existing_times = set(
    str(row["time"])[:16]
    for row in spark.table("my_pipeline.bronze.weather")
        .select("time")
        .collect()
)

new_times = set(t[:16] for t in data["hourly"]["time"])

print("Fetched:", len(data["hourly"]["time"]))
print("New timestamps:", len(new_times - existing_times))


Fetched: 384
New timestamps: 384


In [0]:
import pandas as pd

hourly = data["hourly"]

new_pdf = pd.DataFrame(hourly)
new_pdf["location"] = "Chennai"
new_pdf["latitude"] = data["latitude"]
new_pdf["longitude"] = data["longitude"]
new_pdf["source"] = "Open-Meteo"

new_df = spark.createDataFrame(new_pdf)

print("Prepared rows:", new_df.count())
new_df.printSchema()

Prepared rows: 384
root
 |-- time: string (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- rain: double (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- wind_direction_10m: long (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- location: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source: string (nullable = true)



In [0]:
bronze_df = spark.table("my_pipeline.bronze.weather")

print("Existing Bronze rows:", bronze_df.count())
bronze_df.printSchema()

Existing Bronze rows: 4512
root
 |-- time: string (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: long (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- location: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import current_timestamp

new_df = new_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

new_df = new_df.select(*bronze_df.columns)

print("New rows:", new_df.count())
new_df.printSchema()

New rows: 384
root
 |-- time: string (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: long (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- location: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)



In [0]:
new_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("my_pipeline.bronze.weather")

In [0]:
print(
    "Total Bronze rows:",
    spark.table("my_pipeline.bronze.weather").count()
)

Total Bronze rows: 4896


In [0]:
print("Bronze:", spark.table("my_pipeline.bronze.weather").count())
print("Silver:", spark.table("my_pipeline.silver.weather").count())
print("Gold:", spark.table("my_pipeline.gold.weather").count())

Bronze: 5232
Silver: 4896
Gold: 204
